## Введение

Проект - House Prices - Advanced Regression Techniques.
Цель - построить модель ML для предсказания стоимости домов в Ames, Iowa, используя датасет  Ames Housing dataset.
План:
1. Предварительный анализ данных: посмотреть распределение таргетов, рассмотреть выбросы, дубликаты, пропуски, категориальные переменные + корреляции и зависимости
2. Проверка data leakage
3. Создание валидационного разделения + выбор метрики
4. Baseline:  Dummy regressor/линейная регрессия/дерево решений
5. Preprocessing: пропуски, категориальные переменные, Pipeline + ColumnTransformer
6. Случайный лес + Кросс Валидация + Grid Search
7. Анализ признаков: feature importance, SHAP, поэкспериментировать с feature selection
8. Градиентный бустинг, optuna + кросс-валидация
9. Выбор лучшей модели

В силу условия проекта, будем работать с метрикой RMSE, рассчитанной на основе логарифма прогнозируемого значения и логарифма наблюдаемой цены продажи, будем называть ее RMSEL. Подключим нужные библиотеки.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
import shap

from sklearn.metrics import root_mean_squared_error, make_scorer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

## Загрузка данных, распределение трагетов

Загрузим данные.

In [ ]:
iowa_file_path = '/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv'
test_file_name = '/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv'

train = pd.read_csv(iowa_file_path)
y = train.SalePrice
test = pd.read_csv(test_file_name)

# Разделим признаки на числовые и категориальные 
numeric_cols = train.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = train.select_dtypes(include=["object"]).columns

In [ ]:
train.head()

In [ ]:
train.describe()

Посмотрим на распределение таргетов.

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(train["SalePrice"], kde=True)
plt.xlabel("SalePrice")
plt.title("Distribution of SalePrice")
plt.show()

## Выбросы и дубликаты

Рассмотрим распределение потенциальных выбросов, но удалять предпроложительные выбросы не будем.

In [ ]:
sns.boxplot(y=train["SalePrice"])
plt.show()

In [ ]:
outliers = {}

for col in numeric_cols:
    Q1 = train[col].quantile(0.25)
    Q3 = train[col].quantile(0.75)
    IQR = Q3 - Q1

    mask = (train[col] < Q1 - 1.5 * IQR) | (train[col] > Q3 + 1.5 * IQR)
    outliers[col] = mask.sum()

pd.Series(outliers).sort_values(ascending=False)

In [ ]:
train.duplicated().sum()

Дубликатов нет.

## Preprocessing

### Data leakage

Проверим data leakage. Подозрительные признаки: Id (не подозрительный, но явно лишний), SalePrice. Утечку через target уберем устранив подозрительные признаки. Утечку через preprocessing уберем с помощью pipeline. Удалим подозрительные.

### Missing values

Посмотрим в каких столбцах есть пропуски и сколько их. 

In [ ]:
missing_df = pd.DataFrame({
    "dtype": train.dtypes,
    "count": train.isnull().sum(),
    "percent": train.isnull().mean() * 100
})

missing_df = missing_df[
    missing_df["count"] > 0
].sort_values("percent", ascending=False)

missing_df

Разберемся с пропущенными данными.  
Признаки из train_use, у которых >50% пропусков, впоследствии удалим. Пропусков в типе int64 нет.  
Пропуски LotFrontage заполняем средним. Признаки GarageType, GarageYrBlt, GarageFinish, GarageQual, GarageCond связаны и пропуски, видимо, означают отсутствие гаража. Поэтому выделим для пропусков отдельные значения: для GarageYrBlt - 0, для GarageType, GarageFinish, GarageQual, GarageCond - "None". В MasVnrArea заполним средним и добавим столбец True/False.  
Остались только категориальные переменные FireplaceQu, BsmtExposure, BsmtFinType2, BsmtQual, BsmtCond, BsmtFinType1, Electrical. FireplaceQu, BsmtExposure, BsmtFinType2, BsmtQual, BsmtCond, BsmtFinType1 - заполняем отдельным "None". Electrical - заполняем самым частым. 	


### Categorical values

Далее разберемся с категориальными переменными. 

In [ ]:
categorical_cols

Список тех, которые допускают упорядочивание: LotShape, LandSlope, ExterQual, ExterCond, BsmtQual, BsmtCond, BsmtExposure, BsmtFinType1, BsmtFinType2, HeatingQC, KitchenQual, Functional, FireplaceQu, GarageFinish, GarageQual, GarageCond, PavedDrive, PoolQC, Fence. Эти признаки (точнее те из этого списка, которые не удалили ранее) закодируем с помощью Ordinal Encoding.
Для остальных: если >10 значений принимает - удаляем, иначе - OneHotEncoding.

### Pipeline

Для удобства и избежания некоторых ошибок создадим pipeline. Конвейер будет включать предварительную обработку train_use (в нашем случае: удаление признаков в >50% пропущенных значений, удаление подозрительных признаков, обработка остальных пропущенных значений, обработка категориальных переменных) и описание модели. 

### Processing number values

In [ ]:
# Определяем числовые признаки
numerical_cols_use = train.select_dtypes(include='number').columns.tolist()

# Выделяем признаки в которых больше 50% пропусков
missing_ratio = train[numerical_cols_use].isna().mean()

drop_cols = missing_ratio[missing_ratio > 0.5].index.tolist()
# Лишние признаки
drop_cols += ['Id', 'SalePrice']

numerical_cols_use = [
    col for col in numerical_cols_use
    if col not in drop_cols
]

# LotFrontage -> среднее
lotfrontage_transformer = SimpleImputer(strategy='mean')

# GarageYrBlt -> 0
garageyrblt_transformer = SimpleImputer(strategy='constant', fill_value=0)

# MasVnrArea -> среднее + индикатор пропуска
masvnrarea_transformer = SimpleImputer(strategy='mean', add_indicator=True)

numerical_transformer = ColumnTransformer(
    transformers=[
        ('lotfrontage', lotfrontage_transformer, ['LotFrontage']),
        ('garageyrblt', garageyrblt_transformer, ['GarageYrBlt']),
        ('masvnrarea', masvnrarea_transformer, ['MasVnrArea'])
    ],
    remainder='passthrough'
)

### Processing categorical values 

In [ ]:
# Определяем категориальные признаки
categorical_cols_use = train.select_dtypes(include='object').columns.tolist()

# Electrical → самое частое значение
electrical_imputer = SimpleImputer(strategy='most_frequent')

# Все остальные категориальные → "None"
categorical_imputer = SimpleImputer(
    strategy='constant',
    fill_value='None'
)

# Удаляем признаки, где >50% пропусков
missing_ratio = train[categorical_cols_use].isna().mean()

categorical_cols_use = [
    col for col in categorical_cols_use
    if missing_ratio[col] <= 0.5
]

# Признаки с упорядочиванием
ordinal_cols = ['LotShape','LandSlope','ExterQual','ExterCond','BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2','HeatingQC','KitchenQual','Functional','FireplaceQu','GarageFinish','GarageQual','GarageCond','PavedDrive','PoolQC','Fence']

# Оставляем только те, которые действительно есть после удаления
ordinal_cols = [
    col for col in ordinal_cols
    if col in categorical_cols_use
]

# Остальные категориальные
nominal_cols = [
    col for col in categorical_cols_use
    if col not in ordinal_cols
]

# Убираем Electrical из общего списка
nominal_cols = [
    col for col in nominal_cols
    if col != 'Electrical'
]

# Удаляем номинальные признаки >10 уникальных значений
nominal_cols = [
    col for col in nominal_cols
    if train[col].nunique(dropna=True) <= 5
]

# Шкалы для ordinal-признаков
# Одинаковая шкала для признаков качества
quality_cols = ['ExterQual','ExterCond','BsmtQual','BsmtCond','HeatingQC','KitchenQual','FireplaceQu','GarageQual','GarageCond','PoolQC']

quality_categories = ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex']

# Отдельные шкалы
ordinal_categories_dict = {
    col: quality_categories
    for col in quality_cols
}

ordinal_categories_dict.update({
    'LotShape': ['None', 'IR3', 'IR2', 'IR1', 'Reg'],

    'LandSlope': ['None', 'Sev', 'Mod', 'Gtl'],

    'BsmtExposure': ['None', 'No', 'Mn', 'Av', 'Gd'],

    'BsmtFinType1': [
        'None', 'Unf', 'LwQ', 'Rec',
        'BLQ', 'ALQ', 'GLQ'
    ],

    'BsmtFinType2': [
        'None', 'Unf', 'LwQ', 'Rec',
        'BLQ', 'ALQ', 'GLQ'
    ],

    'Functional': [
        'None', 'Sev', 'Maj2', 'Maj1',
        'Mod', 'Min2', 'Min1', 'Typ'
    ],

    'GarageFinish': ['None', 'Unf', 'RFn', 'Fin'],

    'PavedDrive': ['None', 'N', 'P', 'Y'],

    'Fence': ['None', 'MnWw', 'GdWo', 'MnPrv', 'GdPrv']
})

# Оставляем категории только для тех ordinal-признаков, которые реально остались
ordinal_categories = [
    ordinal_categories_dict[col]
    for col in ordinal_cols
]

onehot_encoder = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=False
)

# Ordinal-признаки
ordinal_transformer = Pipeline([
    ('imputer', categorical_imputer),
    ('encoder', OrdinalEncoder(
        categories=ordinal_categories,
        handle_unknown='use_encoded_value',
        unknown_value=-1
    ))
])

# Electrical
electrical_transformer = Pipeline([
    ('imputer', electrical_imputer),
    ('encoder', onehot_encoder)
])

# Остальные категориальные признаки
nominal_transformer = Pipeline([
    ('imputer', categorical_imputer),
    ('encoder', onehot_encoder)
])

# Объединяем всё
categorical_transformer = ColumnTransformer(
    transformers=[
        ('ordinal', ordinal_transformer, ordinal_cols),
        ('electrical', electrical_transformer, ['Electrical']),
        ('nominal', nominal_transformer, nominal_cols)
    ],
    remainder='drop'
)

In [ ]:
# Общий preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols_use),
        ('cat', categorical_transformer, categorical_cols_use)
    ],
    remainder='drop'
)

In [ ]:
X_processed = preprocessor.fit_transform(train)
feature_names = preprocessor.get_feature_names_out()
train_processed = pd.DataFrame(
    X_processed,
    columns=feature_names,
    index=train.index
)
train_processed.head()

In [ ]:
train_processed.to_csv('train_processed.csv', index=False)

## Baseline

Обучим линейную регрессию. Тут используем обычное разделение выборки на train и valid. Во избежание утечки preprocessing делаем по отдельности для train и valid.

In [ ]:
X = train.drop(columns=['SalePrice'])
y = train['SalePrice']

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

model_lr = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

model_lr.fit(X_train, y_train)

preds_lr = model_lr.predict(X_valid)

rmsel_lr = root_mean_squared_error(
    np.log(y_valid),
    np.log(preds_lr)
)

print(f'RMSEL: {rmsel_lr:.5f}')

## Random Forest

Рассмотрим как меняется ошибка при обучении случайного леса в зависимости от глубины деревьев.

In [ ]:
depths = [5, 10, 15, 20, 25]
errors = []

for depth in depths:
    model_rf = Pipeline([
        ('preprocessor', preprocessor),
        ('model', RandomForestRegressor(
            n_estimators=100,
            max_depth=depth,
            random_state=1
        ))
    ])

    model_rf.fit(X_train, y_train)
    preds_rf = model_rf.predict(X_valid)

    rmsel_rf = root_mean_squared_error(
        np.log(y_valid),
        np.log(preds_rf)
    )

    errors.append(rmsel_rf)

In [ ]:
# Рисуем график
plt.plot(depths, errors, marker='o')
plt.xlabel('max_depth')
plt.ylabel('RMSEL')
plt.title('RMSEL в зависимости от глубины дерева')
plt.grid()
plt.show()

Как можно увидеть, график выходит на плато, поэтому можно использовать depth=15. Заодно посмотрим на важность признаков по SHAP.

In [ ]:
model_rf_1 = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(
        n_estimators=100,
        max_depth=12,
        random_state=1
    ))
])

model_rf_1.fit(X_train, y_train)

X_train_processed = model_rf_1.named_steps['preprocessor'].transform(X_train)

feature_names = (
    model_rf_1
    .named_steps['preprocessor']
    .get_feature_names_out()
)

rf_model = model_rf_1.named_steps['model']

X_shap = X_train_processed[:50]
explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_train_processed)

shap_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': np.abs(shap_values).mean(axis=0)
}).sort_values('importance', ascending=False)

shap_importance.head(10)

In [ ]:
# Рисуем график важности самых важных 10 признаков
top_features = shap_importance.head(10)['feature'].tolist()

top_indices = [i for i, f in enumerate(feature_names) if f in top_features]

shap.summary_plot(
    shap_values[:, top_indices],
    X_train_processed[:, top_indices],
    feature_names=top_features
)

In [ ]:
# Вычисляем ошибку
preds_rf_1 = model_rf_1.predict(X_valid)

rmsel_rf_1 = root_mean_squared_error(
    np.log(y_valid),
    np.log(preds_rf_1)
)
print(f'RMSEL: {rmsel_rf_1:.5f}')

### Cross-validation

Добавим теперь к модели кросс-валидацию.

In [ ]:
model_rf_2 = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(
        n_estimators=100,
        max_depth=12,
        random_state=1
    ))
])

def rmsel(y_true, y_pred):
    return root_mean_squared_error(
        np.log(y_true),
        np.log(y_pred)
    )
    
rmsel_scorer = make_scorer(
    rmsel,
    greater_is_better=False
)

scores = cross_val_score(
    model_rf_2,
    X,
    y,
    cv=5,
    scoring=rmsel_scorer
)

print('Scores:', -scores)
print(f'Mean RMSEL: {-scores.mean():.5f}')

### GridSearch

Теперь улучшим модель с помощью подбора параметоров GridSearch.

In [ ]:
model_rf_3 = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(
        random_state=1
    ))
])

param_grid = {
    'model__n_estimators': [50, 100, 150],
    'model__max_depth': [10, 15, None],
    'model__min_samples_split': [2, 5]
}

grid_search = GridSearchCV(
    model_rf_3,
    param_grid=param_grid,
    cv=5,
    scoring=rmsel_scorer,
    n_jobs=2
)

grid_search.fit(X, y)

print('Best parameters:', grid_search.best_params_)
print(f'Best RMSEL:, {-grid_search.best_score_:.5f}')

## Gradient bossting

Обучим сначала XGBoost без кросс-валидации.

In [ ]:
model_xgb_1 = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(
        n_estimators=250,
        learning_rate=0.05,
        max_depth=6,
        random_state=1
    ))
])

model_xgb_1.fit(X_train, y_train)
preds_xgb_1 = model_xgb_1.predict(X_valid)

rmse_xgb_1 = root_mean_squared_error(
    np.log(y_valid),
    np.log(preds_xgb_1)
)
print(f'RMSEL: {rmse_xgb_1:.5f}')

Теперь добавим кросс-валидацию.

In [ ]:
model_xgb_2 = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(
        n_estimators=250,
        learning_rate=0.05,
        max_depth=6,
        random_state=1
    ))
])

scores = cross_val_score(
    model_xgb_2,
    X,
    y,
    cv=5,
    scoring=rmsel_scorer,
    n_jobs=2
)

print('Scores:', -scores)
print(f'Mean RMSEL: {-scores.mean():.5f}')

## Optuna

Сделаем подбор параметров с помощью Optuna.

In [ ]:
def objective(trial):

    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'max_depth': trial.suggest_int('max_depth', 2, 9),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0)
    }

    model_xgb_3 = Pipeline([
        ('preprocessor', preprocessor),
        ('model', XGBRegressor(
            **params,
            random_state=1
        ))
    ])

    scores = cross_val_score(
        model_xgb_3,
        X,
        y,
        cv=5,
        scoring=rmsel_scorer,
        n_jobs=2
    )

    return -scores.mean()

study = optuna.create_study(
    direction='minimize'
)

study.optimize(
    objective,
    n_trials=7
)

print('Best parameters:',study.best_params)
print(f'Best RMSEL: {study.best_value:.5f}')

Нарисуем график подбора.

In [ ]:
optuna.visualization.plot_optimization_history(study)

## Лучшая модель

Лучшей моделью оказался XGBoost с подбором параметров с помощью Optuna. Поэтому обучим модель на оптимальных параметрах и применим ее к тестовой выборке.

In [ ]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(
        **study.best_params,
        random_state=1
    ))
])

model.fit(X, y)

test_preds = model.predict(test)

In [ ]:
# Записываем результат применения модели в файл
output = pd.DataFrame({'Id': test.Id,
                      'SalePrice': test_preds})
output.to_csv('submission.csv', index=False)